# 03b - The noise floor, and the freeze

**The one sentence for today: yesterday produced three numbers and forbade reading them.
Today measures how much those numbers shake on their own, and only then are they allowed to
be read.**

Day 3 ended with `r` = 4, 8, 16 scoring 0.9988, 0.9987 and 0.9994 on `clean/val`, and with an
explicit refusal to declare a winner. The refusal was right: a spread of 0.0007 means nothing
until something says how much the score moves when *nothing* is changed. That something is
three runs of one configuration differing only in the training seed.

Then, and only then, the configuration is frozen - written to a file, with the rule that chose
it and the result of applying that rule.

**This notebook does not modify `03_train.ipynb`, and it does not re-run the `r` sweep.** It
reads day 3's committed numbers, adds the missing measurement, and writes the decision down.
`clean/test` stays sealed; it opens once, on day 5.

## 0 - Bootstrap and the blocking checks

Identical to day 3, and for the same reason: on a Colab runtime this notebook runs the code
that is **pushed**, not the code open in the editor. The commit is printed so that a run
against a stale clone is visible rather than silent.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/Emma-V/support-triage.git"
BRANCH   = "main"
CLONE_TO = Path("/content/support-triage")


def _git(*args, cwd=None) -> str:
    return subprocess.run(["git", *args], cwd=cwd, check=True,
                          capture_output=True, text=True).stdout.strip()


def _find_repo(start: Path):
    here = start.resolve()
    while not (here / "src" / "data.py").exists():
        if here == here.parent:
            return None
        here = here.parent
    return here


REPO_ROOT = _find_repo(Path.cwd())

if REPO_ROOT is None or REPO_ROOT == CLONE_TO:
    if (CLONE_TO / ".git").exists():
        _git("fetch", "origin", BRANCH, cwd=CLONE_TO)
        _git("reset", "--hard", f"origin/{BRANCH}", cwd=CLONE_TO)
        print(f"updated the existing clone at {CLONE_TO}")
    else:
        _git("clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(CLONE_TO))
        print(f"cloned {REPO_URL} to {CLONE_TO}")
    REPO_ROOT = CLONE_TO

os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))

print(f"\nrepo   {REPO_ROOT}")
print(f"commit {_git('rev-parse', '--short', 'HEAD', cwd=REPO_ROOT)}  "
      f"{_git('log', '-1', '--pretty=%s', cwd=REPO_ROOT)}")
print("\n^ src/noise_floor.py and src/confidence.py are new today. If this commit")
print("  predates them, the imports below fail - push first, then reconnect.")

In [ ]:
# The same two pinned packages as day 3, and the same stale torchao removal.
# Unchanged on purpose: a different library stack would make today's runs
# incomparable to yesterday's, which is the one thing this notebook cannot afford.
import importlib
import importlib.metadata as metadata
import subprocess
import sys

PINNED = {"transformers": "5.14.1", "peft": "0.20.0"}
TORCHAO_MIN_FOR_PEFT = (0, 16)

try:
    have_torchao = metadata.version("torchao")
except metadata.PackageNotFoundError:
    have_torchao = None

stale_torchao = have_torchao is not None and tuple(
    int("".join(c for c in chunk if c.isdigit()) or "0")
    for chunk in have_torchao.split(".")[:2]
) < TORCHAO_MIN_FOR_PEFT

print(f"{'torchao':14s} found {have_torchao or 'nothing':10s} "
      f"{'too old for peft - removing it' if stale_torchao else 'not in the way'}")
if stale_torchao:
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"],
                   check=True)
    importlib.invalidate_caches()

replaced = []
for package, want in PINNED.items():
    try:
        have = metadata.version(package)
    except metadata.PackageNotFoundError:
        have = None
    print(f"{package:14s} found {have or 'nothing':10s} want {want}")
    if have != want:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                        f"{package}=={want}"], check=True)
        replaced.append(f"{package} {have or 'missing'} -> {want}")

if replaced:
    print("\n" + "!" * 70)
    print("REPLACED: " + "; ".join(replaced))
    loaded = [name for name in PINNED if name in sys.modules]
    if loaded:
        print(f"Python is already holding {', '.join(loaded)} in memory. RESTART THE")
        print("KERNEL and run from the top - everything above here is cheap.")
        print("!" * 70)
        raise RuntimeError("restart the kernel: a pinned package was replaced under it")
    print("!" * 70)

import peft
import torch
import transformers

print(f"torch {torch.__version__} | transformers {transformers.__version__} | peft {peft.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if not torch.cuda.is_available():
    print("\n!! No GPU on this runtime. Reconnect and pick a T4 runtime.")

In [ ]:
import gc, json, sys, time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

IN_COLAB = "google.colab" in sys.modules or Path("/content").exists()

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "src" / "data.py").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
assert (REPO_ROOT / "src" / "data.py").exists(), "run the bootstrap cell first"
sys.path.insert(0, str(REPO_ROOT))

from src import confidence as C
from src import data as D
from src import evaluate as E
from src import noise_floor as NF
from src import train as T

METRICS_DIR = REPO_ROOT / "results" / "metrics"
FIGURES_DIR = REPO_ROOT / "results" / "figures"
ARTIFACTS   = REPO_ROOT / "artifacts"
for d in (METRICS_DIR, FIGURES_DIR, ARTIFACTS):
    d.mkdir(parents=True, exist_ok=True)

print("repo:", REPO_ROOT.name, "| colab:", IN_COLAB)
print("seeds today:", NF.SEEDS)

In [ ]:
# Adapters go to Drive, exactly as on day 3. The runtime's own disk does not
# survive the runtime, and a seed run that vanished has to be paid for twice.
DRIVE_OK = False
if IN_COLAB:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        DRIVE_OK = Path("/content/drive/MyDrive").exists()
    except Exception as exc:
        print(f"!! Drive did not mount - {type(exc).__name__}: {exc}")

    if not DRIVE_OK:
        print("\n" + "!" * 70)
        print("RUNNING WITHOUT DRIVE. Day 3's adapters live there, and section 3")
        print("needs run_04_lora_r8 to recover the seed-42 predictions. Without it")
        print("that seed has to be retrained - 47 minutes instead of one.")
        print("!" * 70)

DRIVE_ROOT = (Path("/content/drive/MyDrive/support-triage") if DRIVE_OK
              else Path("/content/_local_runs") if IN_COLAB
              else REPO_ROOT / "_local_runs")
RUNS_DIR = DRIVE_ROOT / "runs"
RUNS_DIR.mkdir(parents=True, exist_ok=True)
print("runs go to:", RUNS_DIR)

### The fast pass, before the expensive one

Same rule as day 3, and the same reason: a smoke pass answers *does this notebook run without
raising* in minutes rather than discovering the answer ninety minutes in. Its outputs go to
`_smoke/` so that nothing it produces can be mistaken for a result three days later.

In [ ]:
SMOKE = True          # <- set to False for the real run

SEEDS       = (42, 43) if SMOKE else NF.SEEDS
SWEEP_EPOCHS = 1 if SMOKE else T.EPOCHS
SWEEP_ROWS   = 512 if SMOKE else None

RUNS_BASE   = RUNS_DIR.parent / "runs" if RUNS_DIR.name == "_smoke" else RUNS_DIR
RUNS_OUT    = RUNS_BASE / "_smoke" if SMOKE else RUNS_BASE
OUT_METRICS = METRICS_DIR / "_smoke" if SMOKE else METRICS_DIR
OUT_FIGURES = FIGURES_DIR / "_smoke" if SMOKE else FIGURES_DIR
for d in (RUNS_OUT, OUT_METRICS, OUT_FIGURES):
    d.mkdir(parents=True, exist_ok=True)

if SMOKE:
    print("SMOKE PASS - a check that the notebook runs, not a result.")
    print(f"  seeds       {SEEDS} instead of {NF.SEEDS}")
    print(f"  epochs      {SWEEP_EPOCHS} instead of {T.EPOCHS}")
    print(f"  train rows  {SWEEP_ROWS} instead of all of clean/train")
    print("  NOTE: a smoke run retrains seed 42 rather than reusing day 3's")
    print("        adapter, because a 512-row model is not that model.")
else:
    print("REAL RUN - three seeds, three epochs, all of clean/train.")
print(f"  writing to  {OUT_METRICS} and {RUNS_OUT}")

In [ ]:
VERSIONS = T.check_transformers_version()
print("transformers", VERSIONS["transformers"], ">=", VERSIONS["min_required"], "OK")

HARDWARE = T.gpu_report()
for k, v in HARDWARE.items():
    print(f"  {k:22s} {v}")
print("\nPRECISION DECIDED:", HARDWARE["precision"], "-", HARDWARE["precision_reason"])

if HARDWARE["gpu_name"] != "Tesla T4":
    print("\n" + "!" * 70)
    print(f"Day 3 ran on a Tesla T4. This runtime is a {HARDWARE['gpu_name']}.")
    print("The seed spread measured here would mix training variance with a")
    print("hardware change, and today's whole point is that only the seed moves.")
    print("Write this down, or reconnect until a T4 is allocated.")
    print("!" * 70)

In [ ]:
manifest = json.loads((REPO_ROOT / "data" / "processed" / "split_manifest.json").read_text())
splits = D.load_all_splits(REPO_ROOT / "data" / "processed")
D.verify_against_manifest(manifest, splits)
print("manifest verified - all six split files match the frozen fingerprints\n")

clean_tr, clean_va = splits["clean"]["train"], splits["clean"]["val"]
INTENTS = json.loads((ARTIFACTS / "labels.json").read_text())
assert len(INTENTS) == 27, len(INTENTS)

print(f"clean/train {len(clean_tr):,} rows   clean/val {len(clean_va):,} rows")
print(f"labels      {len(INTENTS)} intents, frozen order")

# Loaded so the manifest can hash all six files; never read below.
del splits["clean"]["test"], splits["naive"]["test"]

## 1 - The blocking check: were day 3's three runs on the same seed?

Everything today assumes the `r` sweep varied **one** thing. If those three runs also differed
in their training seed, they differ in two variables, and no amount of measurement afterwards
repairs that.

The guide for today says to read the seed out of the run records. **Those records no longer
exist** - they were written into the disposable Colab clone on day 3 and died with the runtime;
only the summary CSVs were recovered, from the notebook's own saved output. So the check is
made against the thing that *is* still authoritative: the committed source of day 3's notebook,
read here as a file rather than remembered.

`RunConfig.train_seed` defaults to `TRAIN_SEED`. If the sweep cell never passed `train_seed`,
then all three runs used the default and the answer is proven rather than assumed.

In [ ]:
nb3 = json.loads((REPO_ROOT / "notebooks" / "03_train.ipynb").read_text(encoding="utf-8"))
sweep_sources = [
    "".join(c["source"]) for c in nb3["cells"]
    if c["cell_type"] == "code" and "T.RunConfig(" in "".join(c["source"])
    and "r sweep" in "".join(c["source"])
]
assert len(sweep_sources) == 1, (
    f"expected exactly one r-sweep cell in 03_train.ipynb, found {len(sweep_sources)}")
sweep_src = sweep_sources[0]

passes_seed = "train_seed" in sweep_src
default_seed = T.RunConfig(name="probe", r=8).train_seed

print("the r-sweep cell as committed in 03_train.ipynb:")
print("-" * 66)
print(sweep_src.strip())
print("-" * 66)
print(f"\n  mentions train_seed anywhere : {passes_seed}")
print(f"  RunConfig default train_seed : {default_seed}")
print(f"  T.TRAIN_SEED                 : {T.TRAIN_SEED}")

assert not passes_seed, (
    "day 3's sweep cell mentions train_seed, so the three r runs may not share "
    "one. Read it above and decide before continuing - this is the blocking check.")
assert default_seed == T.TRAIN_SEED == 42

print(f"\n  [PASS] all three r runs used the default seed {default_seed}.")
print("         The r sweep varied r alone, so today's noise floor can judge it.")

In [ ]:
# The split cannot have moved either, and that is structural rather than lucky:
# nothing here rebuilds the split. The six CSVs are committed, they are hashed
# against split_manifest.json above, and every run reads the same frames.
train_sha = D.sha256_of_split(clean_tr)
manifest_sha = manifest["splits"]["clean"]["train"]["sha256"]
print(f"clean/train sha256 in this runtime : {train_sha[:16]}")
print(f"clean/train sha256 in the manifest : {manifest_sha[:16]}")
assert train_sha == manifest_sha, "the training frame is not the committed one"
print("\n[PASS] the split seed cannot be the thing that moves today - the split is")
print("       not generated at run time, it is read from committed files and hashed.")

sweep_day3 = pd.read_csv(METRICS_DIR / "lora_r_sweep.csv")
print("\nday 3's r sweep, as committed:")
display(sweep_day3)
DAY3_SPREAD = float(sweep_day3["f1_macro_val"].max() - sweep_day3["f1_macro_val"].min())
print(f"spread across r: {DAY3_SPREAD:.4f}   <- the number today has to judge")

## 2 - The decisions, written before anything runs

Recorded here rather than in a comment, because a decision that only exists in somebody's
memory is indistinguishable from a decision made afterwards to fit the result.

**ו1 - the noise floor is measured on `r`=8, the middle value, not on `r`=16, the highest.**
Measuring the spread around the configuration that *won* biases the estimate downward: the
winner is the most likely of the three to have been a lucky draw, and the noise around a lucky
draw looks smaller than it is. A value chosen by a rule fixed in advance - "the middle one" -
is defensible in a way that "the best one" is not. It is also the value most likely to be
chosen in the end if the three turn out indistinguishable.

**ו2 - what the seed moves, and what it does not.** Verified in section 1 rather than trusted.

| the seed changes | the seed does not change |
|---|---|
| classification head initialisation | which rows are in which split - committed, hashed, never rebuilt |
| LoRA matrix initialisation | the training and validation rows themselves |
| shuffling order and batch composition | every hyper-parameter |
| dropout masks during training | the label order in `labels.json` |

**ו3 - if the seeds scatter more widely than the `r` values did, that is the result** and it is
reported as it stands. No fourth seed is run because three looked untidy: `NF.SEEDS` is a
constant in the repository, so adding one shows up as a diff.

**ו5 - the urgency rule already exists.** `artifacts/urgency_prior.json` covers all 27 intents
with no silent default. Today's job is to verify that coverage, not to write it again.

**ו6 - calibration is fitted only if a calibration problem is measured first**, on `clean/val`,
today, while `clean/test` is still sealed. That runs in `03c_confidence.ipynb`, on CPU.

In [ ]:
DECISIONS = {
    "F1_noise_floor_config": f"r={8}, the middle value - not the highest scorer, "
                             "so the noise estimate is not taken around a lucky draw",
    "F2_seed_scope": "train_seed moves head init, LoRA init, shuffling and dropout; "
                     "the split is committed and hash-verified, so it cannot move",
    "F3_no_extra_seeds": f"{len(NF.SEEDS)} seeds fixed in src/noise_floor.py as a constant",
    "F4_freeze_target": "artifacts/config_freeze.json, written today, before the runtime closes",
    "F5_urgency": "artifacts/urgency_prior.json already exists - verified, not rewritten",
    "F6_calibration": "measured before corrected; fitted on clean/val only, while test is sealed",
}
for k, v in DECISIONS.items():
    print(f"  {k:24s} {v}")

NOISE_R = 8
assert NOISE_R in T.R_VALUES and NOISE_R != int(
    sweep_day3.loc[sweep_day3["f1_macro_val"].idxmax(), "r"]), (
    "the noise floor must not be measured on the highest-scoring r - see decision F1")
print(f"\nnoise floor will be measured on r={NOISE_R} "
      f"(alpha={T.LORA_ALPHA_MULTIPLIER * NOISE_R})")

# ו5, verified rather than assumed.
urgency = json.loads((ARTIFACTS / "urgency_prior.json").read_text())
missing = sorted(set(INTENTS) - set(urgency))
extra = sorted(set(urgency) - set(INTENTS))
assert not missing and not extra, f"urgency coverage broken: missing={missing} extra={extra}"
from collections import Counter
print(f"urgency rule: {len(urgency)}/27 intents covered, no silent default, "
      f"levels {dict(Counter(urgency.values()))}")

## 3 - Seed 42: recovered from day 3's adapter, not retrained

Day 3 already trained this exact configuration at seed 42. Its adapter is on Drive; only the
run record was lost. Retraining it would cost 47 minutes and produce the same weights, so
instead the saved adapter is loaded and scored again.

**That re-score is also a check.** Inference is deterministic, so if the recovered macro-F1
comes back as day 3's committed 0.9987, the adapter on Drive is confirmed to be the model that
produced that number. If it does not, something is wrong with the saved adapter and the honest
response is to retrain - not to use it anyway.

The logits are kept this time. They are what day 3 threw away, and they are what makes the
whole confidence and calibration analysis possible without a GPU.

In [ ]:
def score_adapter(adapter_dir, frame, note=""):
    """Load an adapter from disk, score a frame, return metrics + raw logits."""
    model, tokenizer = T.load_adapter(adapter_dir, T.MAIN_MODEL, INTENTS,
                                      HARDWARE["precision"], device=HARDWARE["device"])
    encoded = T.encode_split(tokenizer, frame, INTENTS, HARDWARE["device"])
    metrics, predictions, conf = T.score_encoded(model, encoded, INTENTS,
                                                 HARDWARE["precision"])
    logits = T.predict_logits(model, encoded, precision=HARDWARE["precision"]).numpy()
    del model, tokenizer, encoded
    gc.collect(); torch.cuda.empty_cache()
    print(f"  {note}macro-F1 {metrics['f1_macro']:.4f}  accuracy {metrics['accuracy']:.4f}")
    return metrics, predictions, conf, logits


DAY3_R8 = RUNS_BASE / "run_04_lora_r8"
DAY3_R8_F1 = float(sweep_day3.loc[sweep_day3["r"] == NOISE_R, "f1_macro_val"].iloc[0])
DAY3_R8_EPOCH = int(sweep_day3.loc[sweep_day3["r"] == NOISE_R, "best_epoch"].iloc[0])

REUSE_SEED42 = (not SMOKE) and (DAY3_R8 / "adapter_model.safetensors").exists()
LOGITS = {}
records = []

if REUSE_SEED42:
    print(f"loading day 3's r={NOISE_R} adapter from {DAY3_R8}")
    m42, p42, c42, l42 = score_adapter(DAY3_R8, clean_va, note="recovered: ")
    delta = abs(m42["f1_macro"] - DAY3_R8_F1)
    print(f"\n  day 3 committed : {DAY3_R8_F1:.4f}")
    print(f"  recovered now   : {m42['f1_macro']:.4f}   difference {delta:.6f}")
    if delta > 5e-4:
        raise AssertionError(
            f"the adapter on Drive scores {m42['f1_macro']:.4f}, but day 3 recorded "
            f"{DAY3_R8_F1:.4f}. That is not the model day 3 measured. Set "
            "REUSE_SEED42 = False and retrain seed 42 rather than trusting it.")
    print("\n  [PASS] the adapter on Drive reproduces day 3's number. Seed 42 is recovered,")
    print("         and its 2,120 x 27 logits are kept this time.")
    LOGITS[42] = l42
else:
    print("day 3's adapter is not being reused "
          f"({'smoke pass' if SMOKE else 'not found on Drive'}) - seed 42 will be trained below.")

In [ ]:
if REUSE_SEED42:
    # A record for seed 42 in the same shape the trained runs produce, built from
    # the frozen constants and today's re-score rather than copied from another
    # record - so that the field-by-field comparison in section 5 is a real
    # comparison and not a tautology.
    records.append({
        "name": f"run_04_lora_r{NOISE_R}",
        "config": {
            "base_model": T.MAIN_MODEL, "task_type": "SEQ_CLS",
            "r": NOISE_R, "lora_alpha": T.LORA_ALPHA_MULTIPLIER * NOISE_R,
            "lora_dropout": T.LORA_DROPOUT,
            "target_modules": list(T.TARGET_MODULES),
            "modules_to_save": list(T.MODULES_TO_SAVE),
            "epochs": T.EPOCHS, "best_epoch": DAY3_R8_EPOCH,
            "selection_metric": T.SELECTION_METRIC,
            "learning_rate": T.LEARNING_RATE, "batch_size": T.BATCH_SIZE,
            "grad_accum": T.GRAD_ACCUM, "warmup_ratio": T.WARMUP_RATIO,
            "weight_decay": T.WEIGHT_DECAY, "max_length": D.MAX_LENGTH,
            "precision": HARDWARE["precision"], "gpu_name": HARDWARE["gpu_name"],
            "train_rows": len(clean_tr), "eval_rows": len(clean_va),
            "trained_on": "clean/train", "scored_on": "clean/val",
            "train_sha256": train_sha, "split_seed": D.SPLIT_SEED,
            "train_seed": 42,
        },
        "metrics": m42,
        "runtime_seconds": float(sweep_day3.loc[sweep_day3["r"] == NOISE_R,
                                                "runtime_seconds"].iloc[0]),
        "notes": ("trained on day 3; adapter reloaded from Drive and re-scored today. "
                  "Its provenance differs from the other two seeds, which is why the "
                  "reproduction check above had to pass before it was accepted."),
        "provenance": "day 3 adapter, re-scored",
    })
    print(f"seed 42 record rebuilt from constants + today's re-score "
          f"(macro-F1 {m42['f1_macro']:.4f})")

## 4 - The other seeds

Two runs, or three if seed 42 was not recovered. The only field that differs between them is
`train_seed`; every other value comes from the same module-level constants that day 3 used.

Run numbers continue from day 3 rather than restarting, and no number is ever reused - day 3
ended at `run_05`, so these are `run_06` onward. A run that quietly overwrote another is a run
that has to be done again.

In [ ]:
to_train = [s for s in SEEDS if s not in LOGITS]
print(f"training seeds {to_train} at r={NOISE_R}\n")

for offset, seed in enumerate(to_train):
    config = T.RunConfig(
        name=f"run_{6 + offset:02d}_lora_r{NOISE_R}_seed{seed}",
        r=NOISE_R, train_seed=seed,
        epochs=SWEEP_EPOCHS, train_rows=SWEEP_ROWS,
        notes=f"noise floor, seed {seed}. Only train_seed differs from the other "
              f"runs in this group; r={NOISE_R} was fixed by decision F1 before any "
              "of them ran.")
    print(f"=== {config.name}  (seed={seed}) ===")
    t0 = time.perf_counter()
    out = T.train_one_run(config, clean_tr, clean_va, INTENTS, HARDWARE,
                          RUNS_OUT / config.name)
    out["record"]["provenance"] = "trained today"
    records.append(out["record"])

    # The logits, which day 3 did not keep. One forward pass over the already
    # encoded validation set - the model is still in memory, so it is free here
    # and costs a GPU hour to reproduce later.
    LOGITS[seed] = T.predict_logits(out["model"], out["val_encoded"],
                                    precision=HARDWARE["precision"]).numpy()

    print(f"    finished in {time.perf_counter() - t0:.0f}s  "
          f"macro-F1 {out['record']['metrics']['f1_macro']:.4f}")
    for key in ("model", "tokenizer", "val_encoded"):
        out.pop(key, None)
    gc.collect(); torch.cuda.empty_cache()

print(f"\nall seeds done. logits held for: {sorted(LOGITS)}")

## 5 - The noise floor

Two things before the number: the runs are compared field by field, and their training-data
fingerprints are compared. Remembering that they were identical is not the same as checking.

In [ ]:
same = NF.same_configuration(records)
display(same)

if not same["identical"].all():
    differing = same.loc[~same["identical"], "field"].tolist()
    raise AssertionError(
        f"these runs differ in more than the seed: {differing}. Their spread would "
        "measure that difference as well as training variance, which is not a noise floor.")
print("[PASS] every field except train_seed is identical across the runs.")
print("       train_sha256 matching is the load-bearing one: same rows, same order.")

In [ ]:
seeds_table = NF.seed_table(records)
display(seeds_table)

NOISE = NF.noise_floor(seeds_table["macro-F1 (val)"])
NOISE_ACC = NF.noise_floor(seeds_table["accuracy (val)"])

print(f"macro-F1   mean {NOISE['mean']:.4f}   std {NOISE['std']:.4f}   "
      f"range {NOISE['range']:.4f}")
print(f"accuracy   mean {NOISE_ACC['mean']:.4f}   std {NOISE_ACC['std']:.4f}   "
      f"range {NOISE_ACC['range']:.4f}")
print(f"\nTHE NOISE FLOOR (macro-F1, {NOISE['n']} seeds): {NOISE['std']:.4f}")
print("\nA standard deviation from three points is a rough estimate and is itself")
print("noisy. It supports one sentence - 'a gap smaller than this is not")
print("distinguishable from training variance' - and no significance claim.")

### The second ruler: how much is one validation row worth?

The standard deviation is not the only way to read a gap of 0.0007, and at this accuracy it is
not the clearest one. macro-F1 over 2,120 rows and 27 classes does not move continuously: one
row changing its answer moves one class's recall by `1/support`, and that class is `1/27` of
the average. Below that step there is nothing to measure.

Measuring it converts the whole day into a unit anyone can picture - *the spread across `r` is
worth about N validation rows* - and that sentence needs no statistics to be understood.

In [ ]:
best_seed_run = max(records, key=lambda r: r["metrics"]["f1_macro"])
best_logits = LOGITS[best_seed_run["config"]["train_seed"]]
best_pred = [INTENTS[i] for i in best_logits.argmax(axis=1)]

GRAIN = NF.metric_granularity(clean_va["intent"], best_pred, INTENTS, n_probe=200)
for k, v in GRAIN.items():
    print(f"  {k:20s} {v:.6f}" if isinstance(v, float) else f"  {k:20s} {v}")

per_row = GRAIN["mean_drop_per_row"]
print(f"\none validation row is worth about {per_row:.5f} macro-F1")
print(f"day 3's spread across r ({DAY3_SPREAD:.4f}) is worth "
      f"{DAY3_SPREAD / per_row:.1f} validation rows")
print(f"the between-seed std ({NOISE['std']:.4f}) is worth "
      f"{NOISE['std'] / per_row:.1f} validation rows")

## 6 - The rule, applied

The rule was written on day 3, in the notebook, before any of today's numbers existed:

> If the differences turn out smaller than the noise floor, that is a finding and the choice
> falls to cost: the smallest `r` among those that are not statistically distinguishable.

It lives in `src/noise_floor.decision_rule` so that it is code rather than a recollection, and
it is printed next to its own output below. A rule quoted beside the result it produced cannot
have been invented after seeing that result.

In [ ]:
sweep_for_rule = sweep_day3.rename(columns={"f1_macro_val": "macro-F1 (val)"})
CHOSEN = NF.decision_rule(sweep_for_rule, NOISE["std"])

print("THE RULE (fixed on day 3):")
print(f"  {CHOSEN['rule']}")
print(f"  {CHOSEN['registered']}\n")
print("APPLIED:")
print(f"  gap across r            {CHOSEN['gap']:.4f}")
print(f"  between-seed std        {CHOSEN['std']:.4f}")
print(f"  gap in std units        {CHOSEN['gap_in_std_units']:.2f}")
print(f"  distinguishable         {CHOSEN['distinguishable']}")
print(f"  branch taken            {CHOSEN['branch_taken']}")
print(f"\n  CHOSEN r = {CHOSEN['chosen_r']}")
print(f"\n  {CHOSEN['reason']}")
print(f"\nin plain units: the gap is {CHOSEN['gap'] / per_row:.1f} validation rows out of "
      f"{len(clean_va):,}.")

## 7 - The freeze

From here the hyper-parameters do not move. The freeze is a file, not an intention: it names
the run it came from, every value it fixes, the rule that chose it, and the result of applying
that rule. Anything that changes a hyper-parameter after this point breaks the freeze, and
breaking it has to be written down - runs after that belong to a second configuration.

In [ ]:
frozen_run = next((r for r in records
                   if r["config"]["r"] == CHOSEN["chosen_r"]), None)

if frozen_run is None:
    # The rule picked an r the seed group did not train (it was measured on the
    # middle value by design). The freeze still describes the chosen r, so the
    # record is rebuilt from the same frozen constants with r swapped.
    print(f"the rule chose r={CHOSEN['chosen_r']}, which is not the r the noise floor")
    print(f"was measured on (r={NOISE_R}). Building the freeze from the constants.")
    frozen_run = json.loads(json.dumps(records[0]))
    frozen_run["config"]["r"] = CHOSEN["chosen_r"]
    frozen_run["config"]["lora_alpha"] = T.LORA_ALPHA_MULTIPLIER * CHOSEN["chosen_r"]
    frozen_run["config"]["best_epoch"] = int(
        sweep_day3.loc[sweep_day3["r"] == CHOSEN["chosen_r"], "best_epoch"].iloc[0])
    frozen_run["name"] = f"run_0{3 + list(T.R_VALUES).index(CHOSEN['chosen_r'])}_lora_r{CHOSEN['chosen_r']}"
    frozen_run["notes"] = ("configuration frozen from day 3's r sweep; the noise floor "
                           f"that justified the choice was measured at r={NOISE_R}.")

FREEZE = NF.freeze_record(CHOSEN, frozen_run, manifest, NOISE, seeds=SEEDS)
FREEZE["noise_floor"]["measured_at_r"] = NOISE_R
FREEZE["noise_floor"]["one_val_row_is_worth"] = per_row
FREEZE["smoke"] = SMOKE

target = (ARTIFACTS / "config_freeze.json" if not SMOKE
          else OUT_METRICS / "config_freeze_SMOKE.json")
D.write_json(FREEZE, target)
print(json.dumps(FREEZE, indent=2)[:2000])
print(f"\n... written to {target}")
if SMOKE:
    print("\nSMOKE: this is not the freeze. The real one is written by the real run.")

## 8 - Keeping the logits, so that nothing has to be paid for twice

This is the direct repair of day 3's loss. Every confidence, top-3, coverage and calibration
question is answerable from a `(2120, 27)` array of logits, which is 230 KB. Saving it means
`03c_confidence.ipynb` runs on a laptop with no GPU and no Colab session at all.

Logits rather than probabilities: temperature scaling divides the logits before the softmax,
and at this model's confidence a float32 probability underflows to 0.0, whose log is `-inf`.

In [ ]:
saved = []
for seed, logits in sorted(LOGITS.items()):
    path = OUT_METRICS / f"val_logits_r{NOISE_R}_seed{seed}.npy"
    np.save(path, logits.astype(np.float32))
    saved.append((path, logits.shape))

    frame = C.row_frame(logits, clean_va["intent"], INTENTS,
                        texts=clean_va["instruction"], row_ids=clean_va["row_id"])
    frame.to_csv(OUT_METRICS / f"val_predictions_r{NOISE_R}_seed{seed}.csv", index=False)

for path, shape in saved:
    print(f"  {path.name:44s} {shape}  {path.stat().st_size / 1e3:.0f} KB")

# The seed table and the noise floor themselves, as files rather than as printed output.
seeds_table.to_csv(OUT_METRICS / "seed_noise_floor.csv", index=False)
D.write_json({"macro_f1": NOISE, "accuracy": NOISE_ACC, "granularity": GRAIN,
              "decision": CHOSEN, "measured_at_r": NOISE_R, "seeds": list(SEEDS)},
             OUT_METRICS / "noise_floor.json")
for record in records:
    D.write_json(record, OUT_METRICS / f"{record['name']}.json")
print(f"\n{len(records)} run records + seed table + noise floor written to {OUT_METRICS}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))

# Left: the seeds, with the noise band drawn around their mean.
ax = axes[0]
ax.axhspan(NOISE["mean"] - NOISE["std"], NOISE["mean"] + NOISE["std"],
           color="#2f6f6b", alpha=0.13, label=f"+/- 1 std ({NOISE['std']:.4f})")
ax.axhline(NOISE["mean"], color="#2f6f6b", linewidth=1.2,
           label=f"mean {NOISE['mean']:.4f}")
ax.plot(seeds_table["seed"], seeds_table["macro-F1 (val)"], "o",
        color="#2f6f6b", markersize=9)
ax.set_xlabel("training seed"); ax.set_ylabel("macro-F1 on clean/val")
ax.set_title(f"Same configuration (r={NOISE_R}), only the seed moves")
ax.set_xticks(list(seeds_table["seed"])); ax.legend(fontsize=8); ax.grid(alpha=0.25)

# Right: day 3's r sweep, with today's noise band behind it - the whole point.
ax = axes[1]
mid = sweep_day3["f1_macro_val"].mean()
ax.axhspan(mid - NOISE["std"], mid + NOISE["std"], color="#b45309", alpha=0.13,
           label=f"noise floor, +/- 1 std ({NOISE['std']:.4f})")
ax.plot(sweep_day3["r"], sweep_day3["f1_macro_val"], "o-", color="#b45309",
        markersize=9, linewidth=1.2, label="day 3 r sweep")
ax.set_xscale("log", base=2); ax.set_xticks(sweep_day3["r"])
ax.get_xaxis().set_major_formatter(plt.ScalarFormatter())
ax.set_xlabel("LoRA rank r"); ax.set_ylabel("macro-F1 on clean/val")
ax.set_title("Is the r sweep bigger than its own noise?")
ax.legend(fontsize=8); ax.grid(alpha=0.25)

fig.tight_layout()
fig.savefig(OUT_FIGURES / "08_noise_floor.png", dpi=150, bbox_inches="tight")
plt.show()

## 9 - What was deliberately not done today

- **`clean/test` was not opened.** Every number here is on `clean/val`. It opens once, on day 5.
- **No extra seeds were run** after seeing the spread. Three were fixed in advance, in code.
- **No hyper-parameter was swept.** The freeze is signed; that is the whole point of it.
- **The naive split was not touched.** It enters on day 5, under the frozen configuration.
- **No calibration was applied here.** It is measured first, in `03c`, and corrected only if
  a problem is found - and only on `clean/val`, while the test set is still sealed.

In [ ]:
# Verify the adapters really landed, before the runtime is destroyed.
print("adapters on disk:")
for record in records:
    d = record.get("adapter_dir")
    if not d:
        print(f"  [ -- ]  {record['name']:34s} {record.get('provenance', '')}")
        continue
    weights = Path(d) / "adapter_model.safetensors"
    size = weights.stat().st_size / 1e6 if weights.exists() else 0
    print(f"  [{'OK  ' if size else 'GONE'}]  {record['name']:34s} {size:7.1f} MB  {d}")

if not DRIVE_OK and IN_COLAB:
    print(f"\n!! Drive is not mounted - copy {RUNS_OUT} somewhere permanent NOW.")

if IN_COLAB:
    changed = subprocess.run(["git", "status", "--porcelain", "results", "artifacts"],
                             cwd=REPO_ROOT, capture_output=True, text=True).stdout.strip()
    print("\n" + "!" * 70)
    print("COMMIT THESE BEFORE RUNNING THE NEXT CELL. This is exactly what was lost")
    print("on day 3: the results were written into the clone, the runtime was closed,")
    print("and the run records were gone.")
    print("!" * 70)
    print(changed if changed else "  (nothing new - which on a real run is itself suspicious)")
    print("\n  cd /content/support-triage")
    print('  git add results artifacts && git commit -m "day 4: noise floor and freeze"')
    print("  git push")

In [ ]:
# Only after the cell above shows the files committed. Unassigning destroys the
# machine holding them.
print("Everything is written. Closing the runtime.")

if IN_COLAB:
    try:
        from google.colab import runtime
        runtime.unassign()
    except Exception as exc:
        print(f"\ncould not release the runtime from here - {type(exc).__name__}: {exc}")
        print("Disconnect by hand: kernel picker > Disconnect, or")
        print("https://colab.research.google.com > Runtime > Manage sessions")

### Checklist, and what is next

- [ ] **before connecting: `git push`** - this notebook runs the pushed commit
- [ ] the commit printed by the bootstrap cell includes `src/noise_floor.py` and `src/confidence.py`
- [ ] Drive mounted, and day 3's `run_04_lora_r8` adapter found on it
- [ ] blocking: day 3's three `r` runs proven to share seed 42, from the committed notebook
- [ ] blocking: `clean/train` sha256 matches the manifest
- [ ] the GPU is a T4, as on day 3
- [ ] decisions F1-F6 printed before anything trained
- [ ] the noise floor was measured at `r`=8, not at the highest scorer
- [ ] `same_configuration` passed - only `train_seed` differs
- [ ] mean and standard deviation recorded
- [ ] the decision rule was applied and its output recorded
- [ ] `artifacts/config_freeze.json` written **before** the runtime was closed
- [ ] the logit matrices are committed - this is day 3's mistake, not repeated
- [ ] `results/` and `artifacts/` committed and pushed **from the runtime**
- [ ] the runtime was shut down explicitly

**Next:** `03c_confidence.ipynb` - runs on a laptop, no GPU, on the logits saved above.
Then day 5, which opens `clean/test` once.